# DATA_PROCESSING

Builds the modeling datasets from the merged polarimeter/weather files. Same pipeline as the original DATASET_GEN (Section 3.2), with these changes:

- Fidelity uses unit normalized Stokes vectors for both the reference and the measurement.
- Drift targets look forward: window (t + L, t + L + w] with the original w = 300/600/1200 samples. L is the lead time (0 for the main experiment).
- Savitzky-Golay smoothing is causal (current/past samples only).
- Derivative targets are forward differences and are used by the GLMs only.

Targets per sampling rate:
- REXD / RSTDEV x windows 300/600/1200 x (decimated, DSAVG, DSMAX, DSMAX + SAVGOL) = 24 (all models)
- first / second derivative x (decimated, DSAVG, DSMAX, DSMAX + SAVGOL) = 8 (GLMs only)

Outputs go to data/processed/: one dataset per sampling rate, the 10 Hz fidelity series used by the simulator, DOP statistics, and manifest.json, which the other notebooks read.

### ORDER NOTE:

Because of how these were put together, you need to run through them mostly in order, but with a few exceptions. here's the order I used when generating the data/analyses/results for the paper:
1. DATA_PROCESSING (Run up to the optional non-causal pipeline block)
2. DATA_EXPLORATION (this is mostly just so you can see the data and generate plots with it)
3. GENERAL_LINEAR_MODELS (run everything EXCEPT the last block, which we do later for resid autocorr)
4. NONLINEAR_MODELS (Skip the preprocessing factor tests block for now (until NN notebook is complete). Lead-time section should auto-skip here and is mostly just left in for future work)

Then, once all the data is loaded online (NN_MODELS_COLAB.ipynb, manifest.json, model_30SEC.csv, model_1MIN.csv, and model_2MIN.csv), you can run the colab notebook:

5. NN_MODELS_COLAB (Ensure RUN_LEAD_TIMES is set to False so the lead time tests are skipped in cell 22)

Then, download the results from the "results" directory into the local machine and run:

6. NONLINEAR MODELS (From preprocessing factor tests block, 2nd from the end. If the kernel resets, run only blocks 2-15, 19, and 22 (basically, just skip block 21, the "Run" block).)
7. STATIC_SIMULATION
8. GENERAL_LINEAR_MODELS (again, for the residual autocorrelation stuff the reviewer asked for. If the kernel is reset, just re-run blocks 3, 5, and 7, which should correspond to the imports, config, splits, and metrics, then to cell 25 for the resid autocorr tests)

## Imports

In [1]:
import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view
from scipy.signal import butter, filtfilt, group_delay, savgol_coeffs, savgol_filter, sosfilt, sosfilt_zi

## Configuration

In [6]:
RAW_FILES = {
    "30SEC": Path("data/new/non_interpolated_30s_2023_9_29_NIST_LTS.csv"),
    "1MIN": Path("data/new/non_interpolated_1m_2023_9_29_NIST_LTS.csv"),
    "2MIN": Path("data/new/non_interpolated_2m_2023_9_29_NIST_LTS.csv"),
}
SAMPLING_S = {"30SEC": 30, "1MIN": 60, "2MIN": 120}
TIME_COL = "current_time" # POSIX ms
STOKES_COLS = ["S0", "S1", "S2", "S3"]
POL_SAMPLE_PERIOD_S = 0.1 # 10 Hz

# File with the polarimeter DOP columns, used for DOP statistics only (can skip by setting to None)
DOP_SOURCE_CSV = Path("data/old/df_2023_9_29_NIST_LTS.csv")
DOP_TOLERANCE = 0.05

MIN_STOKES_NORM = 1e-9 # below this the SOP direction is undefined -> NaN fidelity

DRIFT_METRICS = ["REXD", "RSTDEV"]
WINDOW_SAMPLES = [300, 600, 1200]
LEAD_TIMES_S = [0, 120, 300, 600] # 0 = main experiment, others for the lead-time evaluation (lead time not yet used)
VARIANTS = ["DEC", "DSAVG", "DSMAX", "SAVGOL"] # decimated, averaged, max, max + Savitzky-Golay

# Low pass applied at 10 Hz before decimation (DEC targets and the decimated fidelity used for the derivatives), as in the original pipeline but causal. Set to False for plain decimation.
DECIMATION_LOWPASS = False
BUTTERWORTH = {"order": 4, "cutoff_hz": 0.0075, "fs_hz": 10.0, "settling_tolerance": 0.01}
SAVGOL = { # same window/order as the original SAVGOL targets
    "30SEC": {"rollmax_steps": 8, "window": 101},
    "1MIN": {"rollmax_steps": 4, "window": 51},
    "2MIN": {"rollmax_steps": 2, "window": 25},
    "polyorder": 3,
}

SPLIT_FRACTIONS = {"train": 0.70, "validation": 0.15} # rest is test, split by row index
# TSCV robustness check: 24 h initial train, 12 h validation, 12 h step, run on the first 85 % only.
TSCV = {"initial_train_h": 24, "val_h": 12, "step_h": 12, "min_last_val_h": 6}

# Sunset -> sunrise (UNIX s) for the node locations, used for plot shading and the daylight baseline
NIGHT_PERIODS_UTC_S = [
    (1696028100, 1696071780),
    (1696114380, 1696158240),
    (1696200660, 1696244700),
]
DISPLAY_TIMEZONE = "America/New_York"

OUT_DIR = Path("data/processed")
DATASET_PATTERN = "model_{sampling}.csv"
FIDELITY_OUT = OUT_DIR / "fidelity_10hz.csv"
DOP_STATS_OUT = OUT_DIR / "dop_statistics.csv"
MANIFEST_OUT = OUT_DIR / "manifest.json"

# Optional: also write targets made with the old non-causal filters (filtfilt, centered S-G). Not for modeling or analysis, only for comparing against the previous numbers.
GENERATE_OLD_PIPELINE_DATA = False
OLD_PIPELINE_PATTERN = "old_pipeline_NOT_FOR_MODELING_{sampling}.csv"

OUT_DIR.mkdir(parents=True, exist_ok=True)

## Target names and windows

Every target records where its label window starts and ends relative to t. Bin aggregation (DSAVG, DSMAX, and SAVGOL, which is built on DSMAX) spans the weather interval [t, t + delta), so those windows end one sampling period later.

In [7]:
def target_name(metric, window, lead_s, variant):
    name = f"{metric}_FWD_{window}"
    if lead_s:
        name += f"_L{int(lead_s)}S"
    return name if variant == "DEC" else f"{name}_{variant}"


def derivative_name(order, variant):
    name = f"{'d' if order == 1 else 'd2'}_Fidelity_FWD"
    return name if variant == "DEC" else f"{name}_{variant}"


def drift_target_def(metric, window, lead_s, variant, step_s):
    window_s = window * POL_SAMPLE_PERIOD_S
    binned = variant in ("DSAVG", "DSMAX", "SAVGOL")
    return {"name": target_name(metric, window, lead_s, variant), "metric": metric, "window_samples": window, "lead_s": lead_s, "variant": variant, "primary": lead_s == 0, "glm_only": False, "base": target_name(metric, window, 0, variant), "window_start_offset_s": float(lead_s), "window_end_offset_s": float(lead_s + window_s + (step_s if binned else 0))}


def derivative_target_def(order, variant, step_s):
    # first difference uses F at t and t+delta, binned versions add one interval, second difference adds one more step
    steps = order + (1 if variant in ("DSAVG", "DSMAX", "SAVGOL") else 0)
    name = derivative_name(order, variant)
    return {"name": name, "metric": "d_Fidelity" if order == 1 else "d2_Fidelity", "window_samples": None, "lead_s": 0, "variant": variant, "primary": True, "glm_only": True, "base": name, "window_start_offset_s": 0.0, "window_end_offset_s": float(steps * step_s)}

## Fidelity

F = (1 + r_hat . s_hat)/2 with s_hat = (S1,S2,S3)/|(S1,S2,S3)|. The reference r_hat is the first valid sample, normalized the same way. S0 (intensity) is not part of the SOP direction.

In [8]:
def calculate_fidelity(pol):
    s = pol[["S1", "S2", "S3"]].to_numpy(float)
    norm = np.sqrt((s * s).sum(axis=1))
    valid = np.isfinite(norm) & (norm > MIN_STOKES_NORM)
    unit = np.full_like(s, np.nan)
    unit[valid] = s[valid] / norm[valid, None]
    first = int(np.argmax(valid))
    out = pol.copy()
    out["stokes_norm"] = norm
    out["Fidelity"] = 0.5 * (1.0 + unit @ unit[first])
    ref = {"reference_time_ms": int(pol[TIME_COL].iloc[first]), "reference_unit_vector": [float(v) for v in unit[first]], "n_invalid_stokes": int((~valid).sum())}
    return out, ref

## Forward rolling variability

A trailing window at row j covers rows j-w+1 thru j. Shifting by -(w + lead) gives rows i+lead+1 thru i+lead+w at row i, i.e. (t + L, t + L + w]. REXD and RSTDEV are computed as before.

In [9]:
def forward_rolling_targets(fidelity, lead_s):
    lead_rows = int(round(lead_s / POL_SAMPLE_PERIOD_S))
    f = pd.Series(fidelity)
    out = {}
    for w in WINDOW_SAMPLES:
        rexd = f.rolling(window=w, min_periods=1).max() - f.rolling(window=w, min_periods=1).min()
        rstd = f.rolling(window=w).std()
        out[target_name("REXD", w, lead_s, "DEC")] = rexd.shift(-(w + lead_rows)).to_numpy()
        out[target_name("RSTDEV", w, lead_s, "DEC")] = rstd.shift(-(w + lead_rows)).to_numpy()
    return pd.DataFrame(out)

## Causal butterworth lowpass before decimation (Not used, but should I? I dont remember including this in the first manuscript, but it was in one of my notebooks--maybe its from an older version and I just stopped using it at soome point?)

The original pipeline lowpass filtered the 10 Hz series with filtfilt before decimating. Same order and cutoff here, applied forward only with sosfilt. The output is not shifted back to undo the group delay, since that would use future data. The filter starts at steady state for the first value and the settling period is set to NaN.

In [10]:
BW_SOS = butter(BUTTERWORTH["order"], BUTTERWORTH["cutoff_hz"], btype="low", fs=BUTTERWORTH["fs_hz"], output="sos")
BW_BA = butter(BUTTERWORTH["order"], BUTTERWORTH["cutoff_hz"], btype="low", fs=BUTTERWORTH["fs_hz"])

_step = sosfilt(BW_SOS, np.ones(200_000))
_outside = np.where(np.abs(_step - 1.0) > BUTTERWORTH["settling_tolerance"])[0]
BW_SETTLING_SAMPLES = int(_outside[-1] + 1) if len(_outside) else 0
_, _gd = group_delay(BW_BA, w=[1e-5], fs=BUTTERWORTH["fs_hz"])
BW_GROUP_DELAY_S = float(_gd[0] / BUTTERWORTH["fs_hz"])
print(f"Butterworth settling {BW_SETTLING_SAMPLES * POL_SAMPLE_PERIOD_S:.0f} s, group delay ~{BW_GROUP_DELAY_S:.0f} s")


def causal_butterworth(x):
    x = np.asarray(x, float)
    out = np.full_like(x, np.nan)
    valid = np.isfinite(x)
    if not valid.any():
        return out
    first, last = int(np.argmax(valid)), int(len(x) - 1 - np.argmax(valid[::-1]))
    seg = pd.Series(x[first:last + 1]).ffill().to_numpy() # forward-fill only
    y, _ = sosfilt(BW_SOS, seg, zi=sosfilt_zi(BW_SOS) * seg[0])
    y[:BW_SETTLING_SAMPLES] = np.nan
    out[first:last + 1] = y
    out[~valid] = np.nan
    return out

Butterworth settling 221 s, group delay ~55 s


C:\Users\User\AppData\Local\Temp\ipykernel_44244\1359914953.py:7: UserWarning: The filter's denominator is extremely small at frequencies [0.000],             around which a singularity may be present
  _, _gd = group_delay(BW_BA, w=[1e-5], fs=BUTTERWORTH["fs_hz"])


## Resampling to the weather grid (original functions)

In [11]:
def decimate_to_weather_grid(weather_times, fast_df):
    times = np.unique(np.asarray(weather_times, dtype="int64")) # 1MIN has a few repeated weather time stamps
    return pd.DataFrame({TIME_COL: times}).merge(fast_df, on=TIME_COL, how="left")


def aggregate_to_weather_grid(fast_df, weather_times, cols, how):
    """Mean or max over [t_i, t_{i+1}), labeled t_i (DSAVG / DSMAX)."""
    times = np.unique(np.asarray(weather_times, dtype="int64"))
    f = fast_df[[TIME_COL] + cols].dropna(subset=[TIME_COL]).sort_values(TIME_COL)
    cats = pd.cut(f[TIME_COL], bins=times, right=False, labels=times[:-1])
    grouped = getattr(f.groupby(cats, observed=True)[cols], how)()
    grouped.index = grouped.index.astype("int64")
    grouped.index.name = TIME_COL
    return pd.DataFrame({TIME_COL: times}).merge(grouped.reset_index(), on=TIME_COL, how="left")

## Causal Savitzky-Golay

Rolling max on the DSMAX series (as before), then one-sided SG coefficients so the value at t uses only the current and preceding samples. NaN until a full window is available.

In [12]:
def causal_savgol(x, rollmax_steps, window, polyorder):
    m = pd.Series(np.asarray(x, float)).rolling(window=rollmax_steps, min_periods=1).max().to_numpy()
    coeffs = savgol_coeffs(window, polyorder, pos=window - 1, use="dot")
    out = np.full(len(m), np.nan)
    if len(m) >= window:
        out[window - 1:] = sliding_window_view(m, window) @ coeffs
    return out

## Time of day (original)

In [13]:
def add_tod_sin_cos_after_current_time(df):
    out = df.copy()
    ms_per_day = 86_400_000
    tod_ms = (out[TIME_COL].to_numpy(dtype="int64") % ms_per_day).astype("int64")
    angle = 2.0 * np.pi * (tod_ms / ms_per_day)
    out = out.drop(columns=["time_of_day_ms", "tod_sin", "tod_cos"], errors="ignore")
    at = out.columns.get_loc(TIME_COL) + 1
    out.insert(at, "time_of_day_ms", tod_ms)
    out.insert(at + 1, "tod_sin", np.sin(angle))
    out.insert(at + 2, "tod_cos", np.cos(angle))
    return out

## Build datasets

In [14]:
def build_dataset(sampling):
    step_s = SAMPLING_S[sampling]
    raw = pd.read_csv(RAW_FILES[sampling])
    # some input files repeat a few time stamps. keep the first row for each time
    pol = raw[[TIME_COL] + STOKES_COLS].drop_duplicates(subset=TIME_COL).reset_index(drop=True)
    pol, ref = calculate_fidelity(pol)
    weather = raw.drop(columns=STOKES_COLS, errors="ignore").dropna()
    weather = weather.drop_duplicates(subset=TIME_COL).reset_index(drop=True)
    print(f"{sampling}: removed {len(raw) - len(pol)} repeated polarimeter time stamps, {len(raw.drop(columns=STOKES_COLS).dropna()) - len(weather)} repeated weather time stamps")
    weather_times = weather[TIME_COL].to_numpy("int64")
    savgol = SAVGOL[sampling]

    grid = weather.merge(pol[[TIME_COL, "Fidelity"]], on=TIME_COL, how="left")
    target_defs = []

    # drift targets (REXD / RSTDEV) for every lead time
    for lead in LEAD_TIMES_S:
        fwd = forward_rolling_targets(pol["Fidelity"].to_numpy(), lead)
        cols = list(fwd.columns)
        fwd.insert(0, TIME_COL, pol[TIME_COL].to_numpy())
        dec = fwd.copy()
        if DECIMATION_LOWPASS:
            for c in cols:
                dec[c] = causal_butterworth(fwd[c].to_numpy())
        avg = aggregate_to_weather_grid(fwd, weather_times, cols, "mean").rename(columns={c: f"{c}_DSAVG" for c in cols})
        mx = aggregate_to_weather_grid(fwd, weather_times, cols, "max").rename(columns={c: f"{c}_DSMAX" for c in cols})
        for part in (decimate_to_weather_grid(weather_times, dec), avg, mx):
            grid = grid.merge(part, on=TIME_COL, how="left")
        assert len(grid) == len(weather), "merge changed the number of rows (repeated time stamps?)"
        for c in cols:
            grid[f"{c}_SAVGOL"] = causal_savgol(grid[f"{c}_DSMAX"], savgol["rollmax_steps"], savgol["window"], SAVGOL["polyorder"])
        target_defs += [drift_target_def(m, w, lead, v, step_s) for m in DRIFT_METRICS for w in WINDOW_SAMPLES for v in VARIANTS]

    # derivative targets (GLM only): forward differences on the weather grid
    fid = pol[[TIME_COL, "Fidelity"]]
    fid_dec = fid.assign(dec=causal_butterworth(fid["Fidelity"].to_numpy()) if DECIMATION_LOWPASS else fid["Fidelity"])
    bins = grid[[TIME_COL]].merge(decimate_to_weather_grid(weather_times, fid_dec[[TIME_COL, "dec"]]), on=TIME_COL, how="left").merge(aggregate_to_weather_grid(fid, weather_times, ["Fidelity"], "mean").rename(columns={"Fidelity": "avg"}), on=TIME_COL, how="left").merge(aggregate_to_weather_grid(fid, weather_times, ["Fidelity"], "max").rename(columns={"Fidelity": "max"}), on=TIME_COL, how="left")
    for variant, f in (("DEC", bins["dec"]), ("DSAVG", bins["avg"]), ("DSMAX", bins["max"])):
        f = pd.Series(f.to_numpy(float))
        d1 = (f.shift(-1) - f).abs()
        grid[derivative_name(1, variant)] = d1.to_numpy()
        grid[derivative_name(2, variant)] = (d1.shift(-1) - d1).abs().to_numpy()
    for order in (1, 2):
        grid[derivative_name(order, "SAVGOL")] = causal_savgol(grid[derivative_name(order, "DSMAX")], savgol["rollmax_steps"], savgol["window"], SAVGOL["polyorder"])
    target_defs += [derivative_target_def(o, v, step_s) for o in (1, 2) for v in ("DEC", "DSAVG", "DSMAX", "SAVGOL")]

    # Like before, drop rows missing the main decimated/aggregated drift targets, filter start up NaNs and lead-time tail NaNs stay as NaN so all targets share the same rows (and 70/15/15 split)
    required = [t["name"] for t in target_defs if t["primary"] and not t["glm_only"] and t["variant"] in ("DEC", "DSAVG", "DSMAX")]
    grid = grid.dropna(subset=required).reset_index(drop=True)
    grid = add_tod_sin_cos_after_current_time(grid)

    weather_features = [c for c in weather.columns if c != TIME_COL]
    features = ["tod_sin", "tod_cos"] + weather_features # original columns 2:28
    targets = [t["name"] for t in target_defs]
    grid = grid[[TIME_COL, "time_of_day_ms"] + features + ["Fidelity"] + targets]
    return grid, pol, features, target_defs, ref


datasets, references = {}, {}
for sampling in SAMPLING_S:
    df, pol, feature_columns, target_definitions, ref = build_dataset(sampling)
    df.to_csv(OUT_DIR / DATASET_PATTERN.format(sampling=sampling), index=False)
    datasets[sampling], references[sampling] = df, ref
    print(f"{sampling}: {len(df):,} rows, {len(feature_columns)} features, {len(target_definitions)} targets")
    if sampling == next(iter(SAMPLING_S)): # same 10 Hz record in every input file
        pol[[TIME_COL] + STOKES_COLS + ["stokes_norm", "Fidelity"]].to_csv(FIDELITY_OUT, index=False)
        n_out = int(((pol["Fidelity"] < 0) | (pol["Fidelity"] > 1)).sum())
        print(f"  fidelity outside [0, 1] (not clipped): {n_out}, NaN: {int(pol['Fidelity'].isna().sum())}")
    del pol

30SEC: removed 0 repeated polarimeter time stamps, 0 repeated weather time stamps
30SEC: 8,237 rows, 26 features, 104 targets
  fidelity outside [0, 1] (not clipped): 0, NaN: 0
1MIN: removed 9 repeated polarimeter time stamps, 9 repeated weather time stamps
1MIN: 4,119 rows, 26 features, 104 targets
2MIN: removed 0 repeated polarimeter time stamps, 0 repeated weather time stamps
2MIN: 2,060 rows, 26 features, 104 targets


## DOP statistics

Checks the fully polarized assumption behind the fidelity measure.

In [15]:
dop_sources = {"stokes_norm": pd.read_csv(FIDELITY_OUT, usecols=["stokes_norm"])["stokes_norm"]}
if DOP_SOURCE_CSV is not None and DOP_SOURCE_CSV.exists():
    dop_raw = pd.read_csv(DOP_SOURCE_CSV, usecols=["DOP_percent", "DOP_calc"])
    dop_sources["DOP_calc"] = dop_raw["DOP_calc"]
    dop_sources["DOP_percent / 100"] = dop_raw["DOP_percent"] / 100.0

dop_stats = pd.DataFrame([
    {"quantity": name, "n": int(s.notna().sum()), "mean": s.mean(), "std": s.std(), "min": s.min(),
     "q01": s.quantile(0.01), "median": s.median(), "q99": s.quantile(0.99), "max": s.max(),
     f"frac |DOP-1| > {DOP_TOLERANCE}": float((np.abs(s - 1) > DOP_TOLERANCE).mean())}
    for name, s in dop_sources.items()])

dop_stats.to_csv(DOP_STATS_OUT, index=False)
dop_stats

,quantity,n,mean,std,min,q01,median,q99,max,frac |DOP-1| > 0.05
0,stokes_norm,2472354,1.000089,0.000158,0.994379,0.999940,1.000085,1.000251,1.023941,0.000000
1,DOP_calc,2472354,1.000089,0.000158,0.994379,0.999940,1.000085,1.000251,1.023941,0.000000
2,DOP_percent / 100,2472354,1.005547,0.024304,0.000000,0.968597,0.999390,1.054962,1.999969,0.020722


## Splits and manifest

Original 70/15/15 split by row index. The modeling notebooks drop training rows whose label window runs into the validation period (and validation rows that run into the test period).

In [16]:
splits = {}
for sampling, df in datasets.items():
    n = len(df)
    n_tr, n_va = int(n * SPLIT_FRACTIONS["train"]), int(n * SPLIT_FRACTIONS["validation"])
    t = df[TIME_COL].to_numpy()
    splits[sampling] = {"n": n, "n_train": n_tr, "n_validation": n_va, "n_test": n - n_tr - n_va, "validation_start_ms": int(t[n_tr]), "test_start_ms": int(t[n_tr + n_va])}

manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "time_column": TIME_COL,
    "files": {"fidelity_10hz": FIDELITY_OUT.as_posix(), "dop_statistics": DOP_STATS_OUT.as_posix(),
              "datasets": {s: (OUT_DIR / DATASET_PATTERN.format(sampling=s)).as_posix() for s in SAMPLING_S}},
    "sampling_s": SAMPLING_S,
    "feature_columns": feature_columns,
    "targets": target_definitions,
    "splits": splits,
    "tscv": TSCV,
    "fidelity_references": references,
    "butterworth": {**BUTTERWORTH, "settling_samples": BW_SETTLING_SAMPLES, "group_delay_s": BW_GROUP_DELAY_S},
    "savgol": SAVGOL,
    "night_periods_utc_s": NIGHT_PERIODS_UTC_S,
    "display_timezone": DISPLAY_TIMEZONE,
}
MANIFEST_OUT.write_text(json.dumps(manifest, indent=2))
pd.DataFrame(splits).T

,n,n_train,n_validation,n_test,validation_start_ms,test_start_ms
30SEC,8237,5765,1235,1237,1696188150000,1696225200000
1MIN,4119,2883,617,619,1696188180000,1696225200000
2MIN,2060,1442,309,309,1696188240000,1696225320000


## Optional: old (non-causal) pipeline - not for modeling or analysis, but kept in case I needed it for some reason

Recreates the original filtfilt decimation and centered S-G smoothing on the forward targets, written to separate files that no other notebook reads. Only useful for checking how much the causal filters change the results compared with the published version.

In [ ]:
if GENERATE_OLD_PIPELINE_DATA: # proobably dont need this var if I just make this a standalone cell... Maybe remove later
    for sampling in SAMPLING_S:
        raw = pd.read_csv(RAW_FILES[sampling])
        pol, _ = calculate_fidelity(raw[[TIME_COL] + STOKES_COLS].drop_duplicates(subset=TIME_COL).reset_index(drop=True))
        weather_times = raw.drop(columns=STOKES_COLS).dropna()[TIME_COL].drop_duplicates().to_numpy("int64")
        fwd = forward_rolling_targets(pol["Fidelity"].to_numpy(), 0)
        cols = list(fwd.columns)
        old = fwd.copy()
        for c in cols:
            old[f"{c}_FILTFILT"] = filtfilt(*BW_BA, old[c].interpolate(limit_direction="both").to_numpy())
        old.insert(0, TIME_COL, pol[TIME_COL].to_numpy())
        out = decimate_to_weather_grid(weather_times, old[[TIME_COL] + [f"{c}_FILTFILT" for c in cols]])
        mx = aggregate_to_weather_grid(old, weather_times, cols, "max")
        p = SAVGOL[sampling]
        for c in cols:
            x = mx[c].rolling(window=p["rollmax_steps"], min_periods=1).max()
            y = np.full(len(x), np.nan)
            ok = x.notna().to_numpy()
            y[ok] = savgol_filter(x[ok], window_length=p["window"], polyorder=SAVGOL["polyorder"])
            out[f"{c}_SAVGOL_CENTERED"] = y
        out.to_csv(OUT_DIR / OLD_PIPELINE_PATTERN.format(sampling=sampling), index=False)
        print(f"wrote old-pipeline data for {sampling}")